In [1]:
import pandas as pd

df = pd.read_parquet("../data/data50_2025_01.parquet")
df.tail()

,game_id,event,white_elo,black_elo,opening,winner,moves
49908082,49999996,Rated Blitz game,1868,1859,"French Defense: Exchange Variation, Monte Carl...",1,e2e4 e7e6 d2d4 d7d5 e4d5 e6d5 c2c4 g8f6 c4c5 c...
49908083,49999997,Rated Blitz game,1871,1877,Queen's Pawn Game: Accelerated London System,1,d2d4 d7d5 c1f4 c7c6 g1f3 c8f5 e2e3 e7e6 f1d3 f...
49908084,49999998,Rated Blitz game,1238,1327,Caro-Kann Defense: Exchange Variation,2,e2e4 c7c6 d2d4 d7d5 e4d5 c6d5 b1c3 b8c6 g1f3 c...
49908085,49999999,Rated Blitz game,1468,1468,Italian Game: Anti-Fried Liver Defense,2,e2e4 e7e5 g1f3 b8c6 f1c4 h7h6 d2d4 e5d4 f3d4 f...
49908086,50000000,Rated Blitz game,1784,1775,Benoni Defense: Modern Variation,1,d2d4 g8f6 c2c4 c7c5 d4d5 e7e6 d5e6 f7e6 c1g5 f...


In [2]:
df["event"].value_counts()

event
Rated Blitz game                                                  21046103
Rated Bullet game                                                 16867510
Rated Rapid game                                                   6667514
Rated Classical game                                                276915
Rated UltraBullet game                                              230227
                                                                    ...   
Bullet swiss https://lichess.org/swiss/fCq6TSIS                          1
Blitz swiss https://lichess.org/swiss/2UYJF6rB                           1
Rated Rapid tournament https://lichess.org/tournament/F3AL4qUm           1
Rated Blitz tournament https://lichess.org/tournament/ymzdwhl4           1
Rated Blitz tournament https://lichess.org/tournament/6op45MRA           1
Name: count, Length: 26206, dtype: int64

In [3]:
events_to_keep = [
    "Rated Bullet game",
    "Rated Blitz game",
    "Rated Rapid game",
    "Rated Classical game"
]
df = df[df["event"].isin(events_to_keep)]

In [4]:
top10_openings = (
    df["opening"]
    .value_counts()
    .nlargest(10)
    .index
    .tolist()
)

df_top10 = df[df["opening"].isin(top10_openings)].copy()


def first_n_plies(moves: str, n: int) -> str:
    tokens = moves.split()
    return ' '.join(tokens[:n])


df_top10["first_10_moves"] = df_top10["moves"].apply(lambda mv: first_n_plies(mv, n=10))

variation_counts_top10 = (
    df_top10
    .groupby("opening")["first_10_moves"]
    .nunique()
    .reset_index(name="variation_count")
    .sort_values("variation_count")
)

min_var = variation_counts_top10["variation_count"].min()
best_openings_top10 = variation_counts_top10[variation_counts_top10["variation_count"] == min_var]
sample_opening = best_openings_top10["opening"].iloc[0]

In [5]:
print("Top 10 most played Openings - Variation Counts:")
print(variation_counts_top10.to_string(index=False))
print()
print(f"Opening Line with the fewest variations: {sample_opening}")

Top 10 most played Openings - Variation Counts:
                                      opening  variation_count
Scandinavian Defense: Mieses-Kotroc Variation            92373
                             Philidor Defense           152965
             French Defense: Knight Variation           161178
 Queen's Pawn Game: Accelerated London System           174070
                            Caro-Kann Defense           203547
                         Scandinavian Defense           254152
                               Modern Defense           283542
                                 Pirc Defense           296779
                         Van't Kruijs Opening           589337
                            Queen's Pawn Game           668551

Opening Line with the fewest variations: Scandinavian Defense: Mieses-Kotroc Variation


In [6]:
df_moves = df[df["opening"] == sample_opening].copy()
df_moves["avg_elo"] = df_moves[["white_elo", "black_elo"]].mean(axis=1)
df_moves = df_moves[["avg_elo", "moves", "winner"]]
df_moves.head()

,avg_elo,moves,winner
46,1927.5,e2e4 d7d5 e4d5 d8d5 g1f3 d5f3 d1f3 c8g4 f3g4 g...,1
106,1341.5,e2e4 d7d5 e4d5 d8d5 c2c4 d5a5 b1c3 c7c6 d2d4 a...,2
108,1922.0,e2e4 d7d5 e4d5 d8d5 d2d4 c8f5 b1c3 d5a5 f2f4 c...,2
283,1709.5,e2e4 d7d5 e4d5 d8d5 d2d4 d5d6 b1c3 c8f5 f1c4 b...,2
402,1334.0,e2e4 d7d5 e4d5 d8d5 b1c3 d5e5 g1e2 c8g4 d2d4 e...,2


In [7]:
df_moves.to_parquet("../data/moves_2025_01.parquet", index=False)
print("✅ Saved moves data to '../data/moves_2025_01.parquet'")

✅ Saved moves data to '../data/moves_2025_01.parquet'
